# Regresión Lineal Bayesiana — Pago a Conductores de Uber Bogotá (versión con `trips_pool`)

Este cuaderno clona la especificación original de `DriverPayout`, pero reemplaza `trips_express` por `trips_pool` para evaluar una formulación más coherente con el foco operativo de Uber POOL. El objetivo sigue siendo modelar el **pago total a conductores** (`total_driver_payout`) con **PyMC**.

**Variable objetivo:** `total_driver_payout` (pago total a conductores en COP)

**Predictores seleccionados:**
- `total_matches` — viajes emparejados con al menos 1 pasajero adicional
- `trips_pool` — viajes POOL completados
- `rider_cancellations` — cancelaciones por pasajero
- `treat` — 1 si el tiempo de espera es 5 min (tratamiento), 0 si es 2 min
- `commute` — 1 si es hora pico (7-9:40am o 3-5:40pm)

**Variables excluidas:**
- `trips_express` — se reemplaza aquí para probar una narrativa más centrada en POOL
- `total_double_matches` — subconjunto de `total_matches` (multicolinealidad alta)


## 0. Instalación de dependencias (Google Colab)

In [ ]:
# Descomenta si estás en Google Colab
# !pip install pymc arviz openpyxl -q

## 1. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from sklearn.preprocessing import StandardScaler
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# Carpeta de salida para figuras
IMG_DIR = '../img/resultados_uber_pool'
os.makedirs(IMG_DIR, exist_ok=True)

print(f'PyMC versión: {pm.__version__}')
print(f'ArviZ versión: {az.__version__}')

## 1. Carga y EDA

Leemos la base de datos del experimento de Uber en Bogotá. Cada fila corresponde a un período de 2 horas 40 minutos. El experimento manipuló el tiempo máximo de espera para emparejamiento POOL (2 o 5 minutos) con el fin de analizar su impacto operativo.

In [ ]:
# Ajusta la ruta si ejecutas desde Google Colab
DATA_PATH = '../data/BaseUBER.xlsx'

df = pd.read_excel(DATA_PATH)
print(f'Dimensiones: {df.shape}')
print(f'\nTipos de variables:')
print(df.dtypes)
df.head()

### 1.1 Estadísticas descriptivas

In [ ]:
# Vista general de todas las variables numéricas
desc = df.describe().T
desc['cv'] = desc['std'] / desc['mean']  # coeficiente de variación
display(desc.round(2))

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['total_driver_payout'], bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de total_driver_payout')
axes[0].set_xlabel('Pago total conductores (COP)')
axes[0].set_ylabel('Frecuencia')

# QQ-plot manual para verificar normalidad
from scipy import stats
stats.probplot(df['total_driver_payout'], plot=axes[1])
axes[1].set_title('Q-Q Plot — total_driver_payout')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/01_distribucion_objetivo.png', bbox_inches='tight')
plt.show()
print('Guardado: 01_distribucion_objetivo.png')

### 1.2 Matriz de correlaciones

In [ ]:
# Preparamos variables numéricas para la correlación
df['treat_int'] = df['treat'].astype(int)
df['commute_int'] = df['commute'].astype(int)

cols_corr = [
    'total_driver_payout', 'total_matches', 'trips_pool',
    'rider_cancellations', 'treat_int', 'commute_int',
    'trips_express', 'total_double_matches'
]

corr = df[cols_corr].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # triángulo superior
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Matriz de Correlación de Pearson', fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/02_heatmap_correlacion.png', bbox_inches='tight')
plt.show()
print('Guardado: 02_heatmap_correlacion.png')

print('\nCorrelaciones con total_matches:')
print(corr['total_matches'][['trips_pool', 'trips_express', 'total_double_matches']].round(3))


La correlación alta entre `total_matches`, `trips_pool` y `total_double_matches` sigue indicando posible multicolinealidad. Aun así, en esta versión conservamos `trips_pool` para evaluar una formulación más alineada con el servicio POOL y dejamos explícito ese tradeoff entre coherencia sustantiva y parsimonia estadística.


### 1.3 Boxplots por tratamiento y horario

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Por tratamiento (wait_time)
df['wait_label'] = df['treat'].map({True: '5 mins (treat=1)', False: '2 mins (treat=0)'})
sns.boxplot(
    x='wait_label', y='total_driver_payout', data=df,
    palette='Set2', ax=axes[0]
)
axes[0].set_title('Pago conductores por tiempo de espera (treat)')
axes[0].set_xlabel('Grupo')
axes[0].set_ylabel('total_driver_payout (COP)')

# Por hora pico
df['commute_label'] = df['commute'].map({True: 'Hora pico (commute=1)', False: 'No pico (commute=0)'})
sns.boxplot(
    x='commute_label', y='total_driver_payout', data=df,
    palette='muted', ax=axes[1]
)
axes[1].set_title('Pago conductores por horario (commute)')
axes[1].set_xlabel('Grupo')
axes[1].set_ylabel('total_driver_payout (COP)')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/03_boxplots_treat_commute.png', bbox_inches='tight')
plt.show()
print('Guardado: 03_boxplots_treat_commute.png')

### 1.4 Scatter plots de la variable objetivo vs predictores clave

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# total_driver_payout vs total_matches
sns.regplot(
    x='total_matches', y='total_driver_payout', data=df,
    scatter_kws={'alpha': 0.5, 'color': 'steelblue'},
    line_kws={'color': 'red'}, ax=axes[0]
)
axes[0].set_title('Pago conductores vs Total Matches')
axes[0].set_xlabel('total_matches')
axes[0].set_ylabel('total_driver_payout (COP)')

# total_driver_payout vs trips_pool
sns.regplot(
    x='trips_pool', y='total_driver_payout', data=df,
    scatter_kws={'alpha': 0.5, 'color': 'darkorange'},
    line_kws={'color': 'red'}, ax=axes[1]
)
axes[1].set_title('Pago conductores vs Viajes POOL')
axes[1].set_xlabel('trips_pool')
axes[1].set_ylabel('total_driver_payout (COP)')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/04_scatter_predictores.png', bbox_inches='tight')
plt.show()
print('Guardado: 04_scatter_predictores.png')


## 2. Preprocesamiento

Estandarizamos los predictores numéricos para mejorar la eficiencia del muestreo MCMC. La variable objetivo (`total_driver_payout`) **no** se estandariza para mantener la interpretabilidad directa de los coeficientes en pesos colombianos.

In [ ]:
# Variables predictoras seleccionadas (excluimos trips_express y total_double_matches)
predictores_num = ['total_matches', 'trips_pool', 'rider_cancellations']
predictores_bin = ['treat_int', 'commute_int']

# Estandarizar variables numéricas continuas
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(df[predictores_num])
X_num_df = pd.DataFrame(X_num_scaled, columns=[f'{c}_z' for c in predictores_num])

# Combinar con binarias (ya son 0/1, no requieren escala)
X_df = pd.concat([X_num_df, df[predictores_bin].reset_index(drop=True)], axis=1)
y = df['total_driver_payout'].values

print('Variables en el modelo:')
print(X_df.columns.tolist())
print(f'\nMatriz X: {X_df.shape}')
print(f'Vector y: {y.shape}')
print(f'\nEstadísticas de X estandarizado:')
display(X_df.describe().round(3))

# Guardar medias y desviaciones para interpretación posterior
medias_orig = df[predictores_num].mean()
stds_orig = df[predictores_num].std()
print('\nMedias originales de predictores numéricos:')
print(medias_orig.round(2))


## 3. Modelo Bayesiano con PyMC

Especificamos el siguiente modelo de regresión lineal bayesiana:

$$\text{total\_driver\_payout} \sim N(\mu, \sigma^2)$$

$$\mu = \beta_0 + \beta_1 \cdot \text{total\_matches}_z + \beta_2 \cdot \text{trips\_pool}_z + \beta_3 \cdot \text{rider\_cancellations}_z + \beta_4 \cdot \text{treat} + \beta_5 \cdot \text{commute}$$

**Priors débilmente informativos** (no hay datos históricos de Bogotá, usamos priors amplios que no dominen los datos):

| Parámetro | Prior | Justificación |
|-----------|-------|---------------|
| $\beta_0$ | $N(0, 10000^2)$ | Intercepto: cubre rangos posibles del pago promedio |
| $\beta_1, \beta_2, \beta_3$ | $N(0, 1000^2)$ | Efectos de variables continuas estandarizadas |
| $\beta_4, \beta_5$ | $N(0, 500^2)$ | Efectos de variables binarias (más acotados) |
| $\sigma$ | $HalfNormal(1000)$ | Varianza residual positiva |

Los predictores continuos están estandarizados, por lo que los coeficientes representan el cambio en el pago (COP) por cada desviación estándar de cambio en el predictor.


In [ ]:
# Convertir a arrays numpy para PyMC
total_matches_z = X_df['total_matches_z'].values
trips_pool_z = X_df['trips_pool_z'].values
rider_cancellations_z = X_df['rider_cancellations_z'].values
treat_x = X_df['treat_int'].values
commute_x = X_df['commute_int'].values

with pm.Model() as modelo_uber:
    # ---- PRIORS débilmente informativos ----
    beta0 = pm.Normal('beta0', mu=0, sigma=10000)          # intercepto
    beta1 = pm.Normal('beta1_matches', mu=0, sigma=1000)   # total_matches
    beta2 = pm.Normal('beta2_pool', mu=0, sigma=1000)      # trips_pool
    beta3 = pm.Normal('beta3_cancel', mu=0, sigma=1000)    # rider_cancellations
    beta4 = pm.Normal('beta4_treat', mu=0, sigma=500)      # treat (variable experimental)
    beta5 = pm.Normal('beta5_commute', mu=0, sigma=500)    # commute (hora pico)
    sigma = pm.HalfNormal('sigma', sigma=1000)             # desviación estándar residual

    # ---- MODELO LINEAL ----
    mu = (
        beta0
        + beta1 * total_matches_z
        + beta2 * trips_pool_z
        + beta3 * rider_cancellations_z
        + beta4 * treat_x
        + beta5 * commute_x
    )

    # ---- VEROSIMILITUD ----
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)

print('Modelo especificado correctamente.')
print('Variables del modelo:', [rv.name for rv in modelo_uber.basic_RVs])


In [ ]:
# ---- MUESTREO MCMC (NUTS) ----
# draws=2000, tune=20000, 4 chains siguiendo las especificaciones del modelo
# tune alto para garantizar buena exploración dado el espacio de parámetros
with modelo_uber:
    trace = pm.sample(
        draws=2000,
        tune=20000,
        chains=4,
        random_seed=42,
        target_accept=0.9,   # aceptación alta para reducir divergencias
        return_inferencedata=True
    )

print('\n✓ Muestreo completado.')

## 4. Diagnósticos MCMC

Antes de interpretar los resultados, verificamos la convergencia de las cadenas Markov. Los criterios clave son:
- **R-hat ≈ 1.0**: las 4 cadenas convergieron al mismo espacio posterior
- **Trazas estacionarias**: mezcla adecuada sin tendencias
- **Energy plot**: sin diferencias grandes entre transición y marginal de energía

In [ ]:
# ---- TRACEPLOT ----
var_names = ['beta0', 'beta1_matches', 'beta2_pool',
             'beta3_cancel', 'beta4_treat', 'beta5_commute', 'sigma']

ax = az.plot_trace(trace, var_names=var_names, figsize=(14, 14))
plt.suptitle('Traceplot — Diagnóstico de Convergencia', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/05_traceplot.png', bbox_inches='tight')
plt.show()
print('Guardado: 05_traceplot.png')

In [ ]:
# ---- R-HAT ----
resumen = az.summary(trace, var_names=var_names, round_to=4)
print('Tabla de convergencia (R-hat debe estar muy cerca de 1.0):')
display(resumen[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk', 'ess_tail']])

rhat_max = resumen['r_hat'].max()
if rhat_max < 1.01:
    print(f'\n✓ Convergencia excelente: R-hat máximo = {rhat_max:.4f} (< 1.01)')
elif rhat_max < 1.05:
    print(f'\n⚠ Convergencia aceptable: R-hat máximo = {rhat_max:.4f} (< 1.05)')
else:
    print(f'\n✗ ADVERTENCIA: R-hat máximo = {rhat_max:.4f} — revisar modelo o aumentar tune')

In [ ]:
# ---- ENERGY PLOT ----
# Verifica que la distribución de energía de las transiciones y la marginal coincidan.
# Si difieren significativamente, indica problemas de exploración del espacio posterior.
ax = az.plot_energy(trace, figsize=(8, 4))
plt.title('Energy Plot — Exploración del Espacio Posterior')
plt.savefig(f'{IMG_DIR}/06_energy_plot.png', bbox_inches='tight')
plt.show()
print('Guardado: 06_energy_plot.png')

# Verificar divergencias
n_div = int(trace.sample_stats.diverging.sum())
if n_div == 0:
    print(f'✓ Sin divergencias ({n_div})')
else:
    print(f'⚠ Divergencias detectadas: {n_div}')

## 5. Resultados Posteriores

Una vez confirmada la convergencia, analizamos las distribuciones posteriores de los coeficientes. En el marco bayesiano obtenemos **distribuciones completas** (no puntos únicos), lo que permite cuantificar la incertidumbre sobre cada efecto.

In [ ]:
# ---- POSTERIOR DE COEFICIENTES (sin intercepto y sigma para mejor visualización) ----
coef_vars = ['beta1_matches', 'beta2_pool', 'beta3_cancel', 'beta4_treat', 'beta5_commute']

az.plot_posterior(
    trace,
    var_names=coef_vars,
    hdi_prob=0.95,
    figsize=(14, 8),
    textsize=10
)
plt.suptitle('Distribuciones Posteriores de los Coeficientes (HDI 95%)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/07_posterior_coeficientes.png', bbox_inches='tight')
plt.show()
print('Guardado: 07_posterior_coeficientes.png')

In [ ]:
# ---- POSTERIOR DEL INTERCEPTO Y SIGMA ----
az.plot_posterior(
    trace,
    var_names=['beta0', 'sigma'],
    hdi_prob=0.95,
    figsize=(12, 4)
)
plt.suptitle('Distribuciones Posteriores: Intercepto y Sigma', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/08_posterior_beta0_sigma.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- FOREST PLOT (Intervalos de Credibilidad al 95%) ----
# Muestra los HDI de todos los coeficientes en un solo gráfico comparativo
ax = az.plot_forest(
    trace,
    var_names=coef_vars,
    hdi_prob=0.95,
    combined=True,
    figsize=(9, 5)
)
plt.axvline(0, color='red', linestyle='--', linewidth=1.2, label='Efecto nulo')
plt.title('Forest Plot — Intervalos HDI al 95% de los Coeficientes', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/09_forest_plot.png', bbox_inches='tight')
plt.show()
print('Guardado: 09_forest_plot.png')

In [ ]:
# ---- TABLA RESUMEN COMPLETA ----
print('Tabla resumen de la distribución posterior:')
print('(mean=media posterior, sd=desviación, hdi_3%/hdi_97%=intervalo de credibilidad 95%)')
display(resumen)

## 6. Posterior Predictive Check (PPC)

El PPC verifica si el modelo puede generar datos parecidos a los observados. Si las distribuciones predichas cubren bien los datos reales, el modelo está bien especificado. Un mal ajuste sugeriría cambiar la distribución de la verosimilitud o agregar más estructura al modelo.

In [ ]:
# ---- MUESTREO PREDICTIVO POSTERIOR ----
with modelo_uber:
    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

print('✓ Muestreo predictivo posterior completado.')

In [ ]:
# ---- GRÁFICO PPC ----
az.plot_ppc(
    ppc,
    observed=True,
    num_pp_samples=200,
    figsize=(10, 5)
)
plt.title('Posterior Predictive Check — Ajuste del Modelo', fontsize=12)
plt.xlabel('total_driver_payout (COP)')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/10_ppc.png', bbox_inches='tight')
plt.show()
print('Guardado: 10_ppc.png')

In [ ]:
# ---- PREDICHOS vs OBSERVADOS ----
# Media posterior de las predicciones para cada observación
y_pred_mean = ppc.posterior_predictive['y_obs'].mean(dim=['chain', 'draw']).values
y_pred_lower = ppc.posterior_predictive['y_obs'].quantile(0.025, dim=['chain', 'draw']).values
y_pred_upper = ppc.posterior_predictive['y_obs'].quantile(0.975, dim=['chain', 'draw']).values

residuales = y - y_pred_mean

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter predichos vs reales
axes[0].scatter(y, y_pred_mean, alpha=0.6, color='steelblue', edgecolors='white', linewidths=0.3)
lim = [min(y.min(), y_pred_mean.min()) * 0.95, max(y.max(), y_pred_mean.max()) * 1.05]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlim(lim)
axes[0].set_ylim(lim)
axes[0].set_xlabel('Valores reales (COP)')
axes[0].set_ylabel('Media posterior predicha (COP)')
axes[0].set_title('Predichos vs Observados')
axes[0].legend()

# Residuales
axes[1].scatter(y_pred_mean, residuales, alpha=0.6, color='darkorange', edgecolors='white', linewidths=0.3)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Media posterior predicha (COP)')
axes[1].set_ylabel('Residual (COP)')
axes[1].set_title('Residuales vs Predichos')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/11_predichos_vs_reales.png', bbox_inches='tight')
plt.show()
print('Guardado: 11_predichos_vs_reales.png')

In [ ]:
# ---- R² BAYESIANO ----
# az.r2_score espera y_pred con forma (n_samples, n_obs).
# ppc.posterior_predictive['y_obs'] tiene forma (chains, draws, obs), así que
# apilamos chains y draws en una sola dimensión antes de pasarlo.
y_pred_2d = ppc.posterior_predictive['y_obs'].values.reshape(-1, len(y))

r2 = az.r2_score(y, y_pred_2d)
print(f'R² Bayesiano:')
print(f'  Media:  {r2["r2"]:.4f}')
print(f'  SD:     {r2["r2_std"]:.4f}')
print(f'\nInterpretación: el modelo explica en promedio el {r2["r2"]*100:.1f}% de la varianza')
print(f'en el pago total a conductores.')

## 7. Interpretación Automática de Resultados

A continuación se genera automáticamente un resumen interpretativo en español basado en los valores posteriores estimados.

In [ ]:
# Extraer medias posteriores y HDI 95% de los coeficientes
posterior = trace.posterior

def get_stats(var):
    """Retorna (media, hdi_low, hdi_high) de una variable posterior."""
    vals = posterior[var].values.flatten()  # 1D array con todas las muestras
    mean_val = float(vals.mean())
    hdi = az.hdi(vals, hdi_prob=0.95)      # devuelve array 1D shape (2,): [low, high]
    return mean_val, float(hdi[0]), float(hdi[1])

b0_m, b0_l, b0_h   = get_stats('beta0')
b1_m, b1_l, b1_h   = get_stats('beta1_matches')
b2_m, b2_l, b2_h   = get_stats('beta2_pool')
b3_m, b3_l, b3_h   = get_stats('beta3_cancel')
b4_m, b4_l, b4_h   = get_stats('beta4_treat')
b5_m, b5_l, b5_h   = get_stats('beta5_commute')
sig_m, sig_l, sig_h = get_stats('sigma')
r2_val = r2['r2']

def signo(val):
    return 'POSITIVO' if val > 0 else 'NEGATIVO'

def cruza_cero(low, high):
    return low < 0 < high

# Identificar el predictor con mayor efecto absoluto (en escala estandarizada)
efectos = {
    'total_matches':       abs(b1_m),
    'trips_pool':       abs(b2_m),
    'rider_cancellations': abs(b3_m),
    'treat':               abs(b4_m),
    'commute':             abs(b5_m),
}
predictor_mayor = max(efectos, key=efectos.get)

# ---- RESUMEN INTERPRETATIVO ----
print('=' * 70)
print('INTERPRETACIÓN DE RESULTADOS — REGRESIÓN BAYESIANA UBER BOGOTÁ')
print('=' * 70)

print(f"""
1. EFECTO DEL TRATAMIENTO (beta4 — wait_time = 5 mins)
   Media posterior: {b4_m:,.1f} COP  [HDI 95%: {b4_l:,.1f} a {b4_h:,.1f}]
   Signo: {signo(b4_m)}

   El coeficiente beta4_treat es {signo(b4_m).lower()}, lo que significa que pasar
   del tiempo de espera de 2 minutos a 5 minutos {'AUMENTA' if b4_m > 0 else 'REDUCE'}
   el pago total a conductores en aproximadamente {abs(b4_m):,.0f} COP por período.
   {'El HDI cruza el cero, lo que indica que este efecto NO es concluyente (incertidumbre alta).' if cruza_cero(b4_l, b4_h) else 'El HDI NO cruza el cero, lo que indica evidencia sólida de este efecto.'}
   Operativamente: {'esperar más tiempo permite emparejar mejor los viajes, generando más ingresos para conductores.' if b4_m > 0 else 'el mayor tiempo de espera puede desincentivar viajes y reducir el pago total.'}
""")

print(f"""
2. EFECTO DE HORA PICO (beta5 — commute = 1)
   Media posterior: {b5_m:,.1f} COP  [HDI 95%: {b5_l:,.1f} a {b5_h:,.1f}]
   Signo: {signo(b5_m)}

   El coeficiente beta5_commute es {signo(b5_m).lower()}, lo que implica que durante
   horas pico el pago total a conductores {'aumenta' if b5_m > 0 else 'disminuye'}
   en promedio {abs(b5_m):,.0f} COP por período.
   {'El HDI cruza el cero: no hay evidencia concluyente del efecto de la hora pico.' if cruza_cero(b5_l, b5_h) else 'El HDI no cruza el cero: el efecto de la hora pico es estadísticamente robusto.'}
   Operativamente: {'la mayor demanda en hora pico se traduce en más viajes completados y mayor pago a conductores.' if b5_m > 0 else 'durante hora pico podría haber saturación que reduce la eficiencia operativa.'}
""")

print(f"""
3. PREDICTOR CON MAYOR EFECTO (según magnitud de media posterior)
   Variable: {predictor_mayor}
   Efecto absoluto (escala estandarizada): {efectos[predictor_mayor]:,.1f} COP por σ

   En el Forest Plot, '{predictor_mayor}' muestra el intervalo más alejado
   del cero y la mayor magnitud de efecto sobre el pago a conductores.
   Esto confirma que esta variable es el principal motor del pago en el modelo bajo la especificación centrada en viajes POOL.
""")

print(f"""
4. CONCLUSIÓN: ¿ES VIABLE UBER POOL EN BOGOTÁ?
   R² Bayesiano: {r2_val:.3f} ({r2_val*100:.1f}% de varianza explicada)
   Sigma residual: {sig_m:,.0f} COP  [HDI 95%: {sig_l:,.0f} a {sig_h:,.0f}]

   El modelo explica el {r2_val*100:.1f}% de la variación en el pago a conductores,
   lo cual {'es una bondad de ajuste razonable' if r2_val > 0.6 else 'sugiere que hay factores adicionales relevantes no capturados en el modelo'}.

   Desde el punto de vista operativo:
   - beta4_treat {'NO cruza el cero → evidencia de efecto real.' if not cruza_cero(b4_l, b4_h) else 'cruza el cero → efecto incierto.'}
     El aumento del tiempo de espera a 5 minutos {'BENEFICIA a los conductores.' if not cruza_cero(b4_l, b4_h) and b4_m > 0 else 'no genera un beneficio claro para los conductores.'}
   - beta5_commute {'NO cruza el cero → efecto robusto de la hora pico.' if not cruza_cero(b5_l, b5_h) else 'cruza el cero → efecto de hora pico incierto.'}
     Concentrar el servicio en horas pico {'es una estrategia rentable.' if not cruza_cero(b5_l, b5_h) and b5_m > 0 else 'no garantiza mayor rentabilidad.'}

   Conclusión general: Uber POOL muestra {'señales de viabilidad operativa en Bogotá. El modelo sugiere que el diseño experimental (especialmente el tiempo de espera) tiene impacto medible sobre los ingresos de los conductores.' if r2_val > 0.5 else 'una relación compleja entre las variables operativas y el pago a conductores. Se requieren modelos más ricos (efectos de interacción, series de tiempo) para conclusiones definitivas sobre la viabilidad en Bogotá.'}
""")

print('=' * 70)

## Resumen de figuras generadas

Todas las figuras se guardaron en `img/resultados_uber_pool/`:


In [ ]:
import glob
figuras = sorted(glob.glob(f'{IMG_DIR}/*.png'))
print(f'Figuras generadas ({len(figuras)} archivos):')
for f in figuras:
    print(f'  {os.path.basename(f)}')